<font color='red' size='6'><b>01. 데이터 마트 구축</b></font>

- 목적 : 흩어진 상권분석 데이터 11개를 **상권 × 업종 × 분기** 한 행으로 합친 분석용 데이터(마트) 만들기
- WBS : 0.3 상권 매핑 → 1.1 로드·컬럼명 통일 → 1.2 병합
- 결과 : `00_data/processed/mart_상권_업종_분기.csv`

# 1. 패키지로드 & 한글설정 & 경고메세지 ignore

In [ ]:
# 패키지 import
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# 시각화의 선명도를 높임
%config InlineBackend.figure_format = "retina"

# 한글설정 및 미적속성 설정
sns.set(style='white',
        context='notebook',
        palette='Dark2',
        rc={'figure.figsize': (12,3),
            'font.family':'Malgun Gothic',
            'axes.unicode_minus':False})

# warning(경고) 안보이게
import warnings
warnings.filterwarnings(action='ignore')

# 모든 컬럼이 보이도록 (컬럼이 많아 가운데가 ... 으로 숨겨지는 것 방지)
pd.set_option('display.max_columns', None)

# 2. 데이터 파일 확인
- 아래 목록의 파일 이름과 3절의 파일 경로가 다르면 3절의 경로를 수정한다

In [ ]:
%ls "00_data/"

# 3. 데이터 읽어오기
- 모든 파일은 cp949 인코딩
- `low_memory=False` : 파일 전체를 한번에 읽어서 타입을 일관되게 추론

In [ ]:
DATA_DIR = '00_data/'

## 3.1 점포 (독립변수1) : 상권 × 업종 × 분기, 영문 컬럼

In [ ]:
store = pd.read_csv(DATA_DIR + '독립변수1_서울시 상권분석서비스(점포-상권)_2025년.csv',
                    encoding='cp949', low_memory=False)
print(store.shape)
store.head(3)

## 3.2 상권변화지표 (독립변수2) : 상권 × 분기

In [ ]:
change = pd.read_csv(DATA_DIR + '독립변수2_서울시 상권분석서비스(상권변화지표-상권).csv',
                     encoding='cp949', low_memory=False)
print(change.shape)
change.head(3)

## 3.3 추정매출 (독립변수3) : 상권 × 업종 × 분기 / 배후지 매출은 영문 컬럼

In [ ]:
sales = pd.read_csv(DATA_DIR + '독립변수3_서울시_상권분석서비스_추정매출-상권__2025년.csv',
                    encoding='cp949', low_memory=False)
sales_bg = pd.read_csv(DATA_DIR + '독립변수3_선택_서울시_상권분석서비스_추정매출-상권배후지__2025년.csv',
                       encoding='cp949', low_memory=False)
print(sales.shape, sales_bg.shape)

## 3.4 길단위인구(유동인구) (독립변수4) : 상권 × 분기

In [ ]:
flow = pd.read_csv(DATA_DIR + '독립변수4_ 서울시 상권분석서비스(길단위인구-상권).csv',
                   encoding='cp949', low_memory=False)
print(flow.shape)
flow.head(3)

## 3.5 상주인구 (독립변수5~9) : 상권 / 상권배후지 / 서울시 / 자치구 / 행정동

In [ ]:
res      = pd.read_csv(DATA_DIR + '독립변수5_서울시 상권분석서비스(상주인구-상권).csv', encoding='cp949')
res_bg   = pd.read_csv(DATA_DIR + '독립변수6_서울시 상권분석서비스(상주인구-상권배후지).csv', encoding='cp949')
res_seoul= pd.read_csv(DATA_DIR + '독립변수7_서울시 상권분석서비스(상주인구-서울시).csv', encoding='cp949')
res_gu   = pd.read_csv(DATA_DIR + '독립변수8_서울시 상권분석서비스(상주인구-자치구).csv', encoding='cp949')
res_dong = pd.read_csv(DATA_DIR + '독립변수9_서울시 상권분석서비스(상주인구-행정동).csv', encoding='cp949')
print(res.shape, res_bg.shape, res_seoul.shape, res_gu.shape, res_dong.shape)

## 3.6 영역-상권 (독립변수10) : 상권 1곳 = 1행

In [ ]:
area = pd.read_csv(DATA_DIR + '독립변수10_서울시 상권분석서비스(영역-상권).csv', encoding='cp949')
print(area.shape)
area.head(3)

# 4. 컬럼명 통일
## 4.1 점포 : 영문 → 한글

In [ ]:
store_cols = {'stdr_yyqu_cd':'기준_년분기_코드',
              'trdar_se_cd':'상권_구분_코드',
              'trdar_se_cd_nm':'상권_구분_코드_명',
              'trdar_cd':'상권_코드',
              'trdar_cd_nm':'상권_코드_명',
              'svc_induty_cd':'서비스_업종_코드',
              'svc_induty_cd_nm':'서비스_업종_코드_명',
              'stor_co':'점포_수',                        # 일반 점포(프랜차이즈 제외)
              'similr_induty_stor_co':'유사_업종_점포_수',  # = 점포_수 + 프랜차이즈_점포_수
              'opbiz_rt':'개업_율',
              'opbiz_stor_co':'개업_점포_수',
              'clsbiz_rt':'폐업_률',
              'clsbiz_stor_co':'폐업_점포_수',
              'frc_stor_co':'프랜차이즈_점포_수'}
store = store.rename(columns=store_cols)
store.columns

## 4.2 배후지 매출 : 영문 → 한글 (분석에 쓰는 컬럼만)

In [ ]:
sales_bg = sales_bg.rename(columns={'stdr_yyqu_cd':'기준_년분기_코드',
                                    'trdar_cd':'상권_코드',
                                    'svc_induty_cd':'서비스_업종_코드',
                                    'thsmon_selng_amt':'배후지_매출_금액',
                                    'thsmon_selng_co':'배후지_매출_건수'})
sales_bg = sales_bg[['기준_년분기_코드','상권_코드','서비스_업종_코드','배후지_매출_금액','배후지_매출_건수']]
sales_bg.head(3)

## 4.3 추정매출 : 시간대 컬럼명 정리
- `시간대_00~06_매출_금액` → `시간대_00_06_매출_금액`
- 깨진 이름 `시간대_건수~06_매출_건수` → `시간대_00_06_매출_건수`

In [ ]:
time_cols = {}
for start, end in [('00','06'),('06','11'),('11','14'),('14','17'),('17','21'),('21','24')]:
    time_cols[f'시간대_{start}~{end}_매출_금액'] = f'시간대_{start}_{end}_매출_금액'
    time_cols[f'시간대_건수~{end}_매출_건수']    = f'시간대_{start}_{end}_매출_건수'
sales = sales.rename(columns=time_cols)
[col for col in sales.columns if '시간대' in col]

## 4.4 전체 점포 수 만들기
- 이 파일의 `점포_수`는 **프랜차이즈를 뺀 일반 점포**
- 폐업률의 분모는 **전체 점포 수 = 점포_수 + 프랜차이즈_점포_수** (= `유사_업종_점포_수`)

In [ ]:
store['전체_점포_수'] = store['점포_수'] + store['프랜차이즈_점포_수']
# 확인 : 1.0 이면 유사_업종_점포_수와 완전히 같음
(store['유사_업종_점포_수'] == store['전체_점포_수']).mean()

# 5. 상권 매핑표 (상권 → 자치구·행정동·위경도)
- 영역-상권 파일에 자치구·행정동이 이미 있음 → 상권_코드로 병합만 하면 됨
- 좌표는 EPSG:5181(파일 설명에 명시) → 위경도 EPSG:4326 으로 변환 (지도용)

In [ ]:
from pyproj import Transformer
tf = Transformer.from_crs('EPSG:5181', 'EPSG:4326', always_xy=True)
area['경도'], area['위도'] = tf.transform(area['엑스좌표_값'].values, area['와이좌표_값'].values)

mapping = area[['상권_코드','상권_코드_명','상권_구분_코드_명','자치구_코드','자치구_코드_명',
                '행정동_코드','행정동_코드_명','위도','경도','영역_면적']]
print(mapping.shape)
mapping[mapping['상권_코드_명']=='신림역(신림)']      # 관악구, 37.48, 126.93 이면 정상

In [ ]:
# 자치구별 상권 수
mapping['자치구_코드_명'].value_counts().plot.bar()
plt.title('자치구별 상권 수')
plt.show()

In [ ]:
# 위경도 산점도 (상권구분별 색상)
plt.figure(figsize=(8,6))
sns.scatterplot(data=mapping, x='경도', y='위도', hue='상권_구분_코드_명', s=15)
plt.title('서울시 상권 위치')
plt.show()

# 6. 병합 → 상권 × 업종 × 분기
- 기준 데이터 : 점포 (`how='left'` : 점포 행은 모두 남기고 옆에 붙임)
- 상권 단위 파일은 이름 컬럼(상권_코드_명 등)을 빼고 붙여서 `_x`, `_y` 중복을 막는다

In [ ]:
key3 = ['기준_년분기_코드','상권_코드','서비스_업종_코드']   # 상권×업종×분기
key2 = ['기준_년분기_코드','상권_코드']                     # 상권×분기
name_cols = ['상권_구분_코드','상권_구분_코드_명','상권_코드_명','서비스_업종_코드_명']

## 6.1 점포 + 추정매출

In [ ]:
mart = store.copy()
print('병합 전 :', mart.shape)
mart = mart.merge(sales.drop(columns=name_cols), on=key3, how='left')
print('병합 후 :', mart.shape)

## 6.2 + 배후지 매출

In [ ]:
mart = mart.merge(sales_bg, on=key3, how='left')
mart.shape

## 6.3 + 유동인구, 상주인구, 상권변화지표 (상권 × 분기)

In [ ]:
mart = mart.merge(flow.drop(columns=['상권_구분_코드','상권_구분_코드_명','상권_코드_명']), on=key2, how='left')
mart = mart.merge(res.drop(columns=['상권_구분_코드','상권_구분_코드_명','상권_코드_명']), on=key2, how='left')
mart = mart.merge(change.drop(columns=['상권_구분_코드','상권_구분_코드_명','상권_코드_명']), on=key2, how='left')
mart.shape

## 6.4 + 배후지 상주인구 (골목상권만 제공)
- `상권배후지_코드` = `상권_코드`

In [ ]:
res_bg2 = res_bg.rename(columns={'상권배후지_코드':'상권_코드',
                                 '총_상주인구_수':'배후지_총_상주인구_수',
                                 '총_가구_수':'배후지_총_가구_수'})
res_bg2 = res_bg2[['기준_년분기_코드','상권_코드','배후지_총_상주인구_수','배후지_총_가구_수']]
mart = mart.merge(res_bg2, on=key2, how='left')
mart.shape

## 6.5 + 상권 매핑 (자치구·행정동·위경도)

In [ ]:
mart = mart.merge(mapping.drop(columns=['상권_코드_명','상권_구분_코드_명']), on='상권_코드', how='left')
mart.shape

## 6.6 + 자치구·행정동 상주인구
- 6.5에서 붙은 `자치구_코드`, `행정동_코드`로 연결

In [ ]:
gu = res_gu[['기준_년분기_코드','자치구_코드','총_상주인구_수','총_가구_수']]
gu = gu.rename(columns={'총_상주인구_수':'자치구_총_상주인구_수', '총_가구_수':'자치구_총_가구_수'})

dong = res_dong[['기준_년분기_코드','행정동_코드','총_상주인구_수','총_가구_수']]
dong = dong.rename(columns={'총_상주인구_수':'행정동_총_상주인구_수', '총_가구_수':'행정동_총_가구_수'})

mart = mart.merge(gu, on=['기준_년분기_코드','자치구_코드'], how='left')
mart = mart.merge(dong, on=['기준_년분기_코드','행정동_코드'], how='left')
mart.shape

## 6.7 사용하지 않을 컬럼 제거
- `아파트_가구_수` : 모든 행이 0 / `비_아파트_가구_수` : 총_가구_수와 같음

In [ ]:
mart = mart.drop(columns=['아파트_가구_수','비_아파트_가구_수'])
mart.shape

# 7. 검증
- 행 수가 점포 파일과 같아야 함 (304,775)
- 상권×업종×분기 키 중복 0

In [ ]:
print('점포 행 수 :', len(store), '/ 마트 행 수 :', len(mart))
print('키 중복 :', mart.duplicated(key3).sum())

In [ ]:
# 결합률 : 붙인 데이터가 몇 % 행에 채워졌는지
check_cols = {'매출':'당월_매출_금액',
              '배후지 매출':'배후지_매출_금액',
              '유동인구':'총_유동인구_수',
              '상주인구':'총_상주인구_수',
              '상권변화지표':'상권_변화_지표',
              '배후지 상주':'배후지_총_상주인구_수',
              '자치구 매핑':'자치구_코드_명',
              '자치구 상주':'자치구_총_상주인구_수',
              '행정동 상주':'행정동_총_상주인구_수'}
join_rate = pd.Series({name: mart[col].notna().mean()*100 for name, col in check_cols.items()})
join_rate.round(1)

In [ ]:
join_rate.sort_values().plot.barh(figsize=(8,4))
plt.title('데이터별 결합률(%)')
plt.xlim(0,100)
plt.show()

- 매출 약 28% : 100개 업종 중 **62개 업종만 매출 제공** → 정상 (02 정제 단계에서 매출 있는 행만 사용)
- 배후지 매출·상주 : **골목상권만 제공** → 정상

# 8. 저장

In [ ]:
os.makedirs(DATA_DIR + 'processed', exist_ok=True)   # 폴더가 없으면 생성
mapping.to_csv(DATA_DIR + 'processed/상권_매핑.csv', index=False, encoding='utf-8-sig')
mart.to_csv(DATA_DIR + 'processed/mart_상권_업종_분기.csv', index=False, encoding='utf-8-sig')
print('저장 완료 :', mart.shape)